# Prithvi-100M Change Detection — Kaggle Training Notebook

Fine-tunes the IBM/NASA Prithvi-100M geospatial ViT for deforestation monitoring over Rondônia, Brazil.

## Setup instructions
1. Upload your tiled dataset to Kaggle:
   - Run the local data pipeline: `python -m data_pipeline.run_pipeline`
   - Go to **kaggle.com/datasets/new** → upload the `data_pipeline/tiles/` folder
   - Name it: `prithvi-rondonia-tiles`
2. Create a new notebook on Kaggle → **File → Add Input** → attach `prithvi-rondonia-tiles`
3. Enable GPU: **Settings → Accelerator → GPU T4 x2** (or P100)
4. Run all cells. Download `best_model.pth` when done.
5. Place `best_model.pth` at `training/checkpoints/best_model.pth` locally.

In [ ]:
# Install dependencies not pre-installed on Kaggle
!pip install timm==0.9.16 huggingface_hub einops --quiet

In [ ]:
import os, sys, json, time, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score, jaccard_score

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
TILES_DIR   = Path('/kaggle/input/prithvi-rondonia-tiles')
OUTPUT_DIR  = Path('/kaggle/working')
CKPT_DIR    = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# Verify dataset is mounted
meta_path = TILES_DIR / 'metadata.json'
assert meta_path.exists(), f'Dataset not found at {TILES_DIR}'
with open(meta_path) as f:
    meta = json.load(f)
print(f"Tiles found: {meta['n_tiles']}  |  size: {meta['tile_size']}px  |  bands: {meta['bands']}")

In [ ]:
# ── Prithvi normalization statistics (from model training config) ────────────
PRITHVI_MEANS = np.array([0.033349706741586264, 0.04509548172188006,
                           0.04026548542688139,  0.26531564765613897,
                           0.16682069560609084,  0.11736093569701502], dtype=np.float32)
PRITHVI_STDS  = np.array([0.02387963425027755,  0.03260987824447789,
                           0.03660491725066752,  0.06836665490453512,
                           0.06832692218471482,  0.05860916013047283], dtype=np.float32)

def denormalize(tile):
    means = PRITHVI_MEANS[:, None, None]
    stds  = PRITHVI_STDS[:, None, None]
    return tile * stds + means

def ndvi(tile):
    raw = denormalize(tile)
    red, nir = raw[2], raw[3]
    d = nir + red
    return np.where(d > 1e-6, (nir - red) / d, 0.0).astype(np.float32)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
FOREST_THRESH = 0.45

class ChangeDataset(Dataset):
    def __init__(self, tiles_dir, indices, augment=False):
        self.tiles_dir = Path(tiles_dir)
        self.augment   = augment
        with open(self.tiles_dir / 'metadata.json') as f:
            meta = json.load(f)
        all_files = [t['filename'] for t in meta['tiles']]
        self.files = [all_files[i] for i in indices]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        fname  = self.files[i]
        before = np.load(self.tiles_dir / 'before' / fname)
        after  = np.load(self.tiles_dir / 'after'  / fname)

        # NDVI pseudo-label: was forest → now not forest = deforestation
        label = ((ndvi(before) >= FOREST_THRESH) &
                 (ndvi(after)  <  FOREST_THRESH)).astype(np.float32)

        if self.augment and np.random.rand() > 0.5:
            before = before[:, :, ::-1].copy()
            after  = after [:, :, ::-1].copy()
            label  = label [:, ::-1].copy()

        return (torch.from_numpy(before),
                torch.from_numpy(after),
                torch.from_numpy(label).unsqueeze(0))

# Train / val split
n     = meta['n_tiles']
n_tr  = int(n * 0.8)
idxs  = np.random.default_rng(42).permutation(n)
train_ds = ChangeDataset(TILES_DIR, idxs[:n_tr], augment=True)
val_ds   = ChangeDataset(TILES_DIR, idxs[n_tr:], augment=False)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
import timm
from huggingface_hub import hf_hub_download

class PrithviEncoder(nn.Module):
    PATCH_GRID = 14
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch16_224', in_chans=6,
            num_classes=0, img_size=224, pretrained=False
        )
    def load_weights(self, path):
        raw = torch.load(path, map_location='cpu', weights_only=False)
        sd  = raw.get('model', raw)
        cleaned = {}
        for k, v in sd.items():
            if 'decoder' in k: continue
            for px in ('encoder.', 'model.', 'module.'):
                if k.startswith(px): k = k[len(px):]; break
            cleaned[k] = v
        exp = self.backbone.pos_embed.shape
        if 'pos_embed' in cleaned and cleaned['pos_embed'].shape != exp:
            pe = cleaned['pos_embed']
            cleaned['pos_embed'] = torch.cat([pe[:,:1,:], pe[:,1:exp[1],:]], 1)
        miss, _ = self.backbone.load_state_dict(cleaned, strict=False)
        print(f'  Loaded {len(cleaned)-len(miss)}/{len(cleaned)} keys')
    def freeze(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)
        self.backbone.eval()
    def forward(self, x):
        f = self.backbone.forward_features(x)[:,1:,:]  # (B,196,768)
        B, N, D = f.shape
        return f.transpose(1,2).reshape(B, D, self.PATCH_GRID, self.PATCH_GRID)

class ChangeHead(nn.Module):
    def __init__(self):
        super().__init__()
        def up(ci, co): return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(ci, co, 3, padding=1, bias=False),
            nn.BatchNorm2d(co), nn.ReLU(inplace=True))
        self.proj = nn.Conv2d(768, 256, 1)
        self.ups  = nn.Sequential(up(256,128), up(128,64), up(64,32), up(32,16))
        self.out  = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        return self.out(self.ups(self.proj(x)))

class ChangeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PrithviEncoder()
        self.head     = ChangeHead()
    def train(self, mode=True):
        super().train(mode)
        if mode: self.encoder.backbone.eval()  # keep backbone in eval
        return self
    def forward(self, b, a):
        fb = self.encoder(b); fa = self.encoder(a)
        return self.head(torch.abs(fa - fb))

# Download & load weights
print('Downloading Prithvi-100M weights...')
ckpt = hf_hub_download('ibm-nasa-geospatial/Prithvi-EO-1.0-100M', 'Prithvi_100M.pt')
model = ChangeDetector()
model.encoder.load_weights(ckpt)
model.encoder.freeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,}  /  Total: {total:,}')

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS     = 30
BATCH_SIZE = 16   # T4 can handle 16; reduce to 8 if OOM
LR         = 1e-3
POS_WEIGHT = 5.0  # deforestation pixels are rare

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
pw        = torch.tensor([POS_WEIGHT], device=device)
criterion = lambda logits, labels: nn.functional.binary_cross_entropy_with_logits(
    logits, labels, pos_weight=pw)
scaler    = torch.cuda.amp.GradScaler()

log, best_f1 = [], 0.0

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_loss = 0.0
    for before, after, labels in train_loader:
        before, after, labels = before.to(device), after.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.autocast('cuda'):
            logits = model(before, after)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()

    model.eval()
    val_loss = 0.0; all_p, all_l = [], []
    with torch.no_grad():
        for before, after, labels in val_loader:
            before, after, labels = before.to(device), after.to(device), labels.to(device)
            logits = model(before, after)
            val_loss += criterion(logits, labels).item()
            preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().flatten()
            all_p.append(preds); all_l.append(labels.cpu().numpy().flatten())

    all_p = np.concatenate(all_p); all_l = np.concatenate(all_l).astype(np.uint8)
    f1  = f1_score(all_l, all_p, zero_division=0)
    iou = jaccard_score(all_l, all_p, zero_division=0)
    scheduler.step()

    row = {'epoch': epoch,
           'train_loss': tr_loss/len(train_loader),
           'val_loss':   val_loss/len(val_loader),
           'f1': f1, 'iou': iou}
    log.append(row)
    print(f"Epoch {epoch:3d} | train={row['train_loss']:.4f} | val={row['val_loss']:.4f} "
          f"| F1={f1:.4f} | IoU={iou:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), CKPT_DIR / 'best_model.pth')
        print(f'  ✓ Saved best (F1={best_f1:.4f})')

torch.save(model.state_dict(), CKPT_DIR / 'last_model.pth')
with open(CKPT_DIR / 'train_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print(f'\nBest val F1 = {best_f1:.4f}')

In [ ]:
# ── Quick loss curve ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x   = [r['epoch']      for r in log]
train_loss = [r['train_loss'] for r in log]
val_f1     = [r['f1']         for r in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_x, train_loss, label='train loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss'); ax1.set_title('Training Loss')
ax2.plot(epochs_x, val_f1,    label='val F1', color='orange')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1'); ax2.set_title('Validation F1')
for ax in (ax1, ax2): ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_curve.png', dpi=120)
plt.show()
print('Saved training_curve.png')

## Download outputs

After the notebook finishes, download these files from `/kaggle/working/checkpoints/`:

| File | Use |
|---|---|
| `best_model.pth` | Best checkpoint by val F1 — use for inference |
| `last_model.pth` | Final epoch checkpoint |
| `train_log.json` | Per-epoch loss and metrics |
| `training_curve.png` | Loss / F1 plots |

Place `best_model.pth` at `training/checkpoints/best_model.pth` in the local repo (gitignored).